# Student AI Impact Analysis
This notebook provides a detailed analysis of the impact of Generative AI tools on university students' academic performance and well-being.

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, r2_score

# Load dataset
df = pd.read_csv('impact_study.csv')

# Display first few rows and basic info
print(df.info())
df.head()

## 2. Univariate Analysis
Exploring individual variables to understand their distributions.

In [ ]:
# Basic statistical summary
print(df.describe())

# Visualization of key numerical variables
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

sns.histplot(df['Semester_GPA'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Semester GPA')

sns.histplot(df['Academic_AI_Hours_Weekly'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Weekly Academic AI Usage Hours')

sns.boxplot(x=df['Age'], ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Age Distribution')

sns.histplot(df['Time_Saved_Pct'], kde=True, ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Time Saved Percentage')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical variables distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.countplot(data=df, y='University', ax=axes[0, 0], hue='University', palette='viridis', legend=False)
axes[0, 0].set_title('Student Count by University')

sns.countplot(data=df, x='Gender', ax=axes[0, 1], hue='Gender', palette='pastel', legend=False)
axes[0, 1].set_title('Gender Distribution')

sns.countplot(data=df, y='Primary_AI_Tool', ax=axes[1, 0], hue='Primary_AI_Tool', palette='magma', legend=False)
axes[1, 0].set_title('Primary AI Tool Preference')

sns.countplot(data=df, x='Urban_Rural', ax=axes[1, 1], hue='Urban_Rural', palette='Set2', legend=False)
axes[1, 1].set_title('Urban vs Rural Students')

plt.tight_layout()
plt.show()

## 3. Bivariate Analysis
Analyzing relationships between variables, specifically focusing on factors affecting GPA.

In [ ]:
# Correlation Matrix of numerical features
plt.figure(figsize=(14, 10))
numerical_df = df.select_dtypes(include=[np.number])
sns.heatmap(numerical_df.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# AI Usage Skills vs GPA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='Literacy_Usage_Skill', y='Semester_GPA', data=df, ax=axes[0], hue='Literacy_Usage_Skill', palette='Set3', legend=False)
axes[0].set_title('Literacy Usage Skill vs GPA')

sns.boxplot(x='Hallucination_Detection', y='Semester_GPA', data=df, ax=axes[1], hue='Hallucination_Detection', palette='Set3', legend=False)
axes[1].set_title('Hallucination Detection Skill vs GPA')

plt.show()

In [ ]:
# Primary AI Tool Impact on GPA
plt.figure(figsize=(12, 6))
sns.barplot(x='Primary_AI_Tool', y='Semester_GPA', data=df, hue='Primary_AI_Tool', palette='muted', legend=False)
plt.title('Average GPA by Primary AI Tool')
plt.xticks(rotation=45)
plt.show()

## 4. Machine Learning: GPA Prediction
Building a model to predict student GPA based on their characteristics and AI usage patterns.

In [ ]:
# Preprocessing
ml_df = df.copy()
le = LabelEncoder()

categorical_cols = ['University', 'Gender', 'Urban_Rural', 'Family_Income', 'Primary_AI_Tool', 'Policy_Preference']
for col in categorical_cols:
    ml_df[col] = le.fit_transform(ml_df[col])

X = ml_df.drop(['Student_ID', 'Semester_GPA'], axis=1)
y = ml_df['Semester_GPA']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Training
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)

# Evaluation
y_pred = rf_reg.predict(X_test)
print(f'R2 Score: {r2_score(y_test, y_pred):.4f}')
print(f'Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.4f}')

# Feature Importance
importances = pd.Series(rf_reg.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x=importances.values[:15], y=importances.index[:15], hue=importances.index[:15], palette='flare', legend=False)
plt.title('Top 15 Predictors of Semester GPA')
plt.show()

## 5. Machine Learning: Student Segmentation
Categorizing students into groups based on their AI usage and stress levels.

In [ ]:
# Selecting features for clustering
cluster_features = ['AI_Daily_Hours', 'Academic_AI_Hours_Weekly', 'Technostress_Overload', 'Literacy_Usage_Skill', 'Semester_GPA']
X_cluster = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-Means Clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Segment'] = kmeans.fit_predict(X_scaled)

# Visualizing Segments
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Academic_AI_Hours_Weekly', y='Semester_GPA', hue='Segment', data=df, palette='bright', alpha=0.6)
plt.title('Student Segments: AI Usage vs GPA')
plt.show()

# Segment Characteristics
segment_summary = df.groupby('Segment')[cluster_features].mean()
print("Mean values for each segment:")
print(segment_summary)

## 6. Conclusions
Summarizing the findings from the analysis.

In [ ]:
print("Analysis Complete.")
print(f"Average GPA across dataset: {df['Semester_GPA'].mean():.2f}")
print(f"Percentage of students using AI: {df['Current_AI_User'].mean()*100:.1f}%")